In [1]:
import sys
import asyncio

# Fix for Windows issues in Jupyter notebooks
if sys.platform == "win32":
    # 1. Use ProactorEventLoop for subprocess support
    if not isinstance(asyncio.get_event_loop_policy(), asyncio.WindowsProactorEventLoopPolicy):
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
    
    # 2. Redirect stderr to avoid fileno() error when launching MCP servers
    if "ipykernel" in sys.modules:
        sys.stderr = sys.__stderr__

C:\Users\Najeeb\AppData\Local\Temp\ipykernel_15244\2246380887.py:7: DeprecationWarning: 'asyncio.get_event_loop_policy' is deprecated and slated for removal in Python 3.16
  if not isinstance(asyncio.get_event_loop_policy(), asyncio.WindowsProactorEventLoopPolicy):
C:\Users\Najeeb\AppData\Local\Temp\ipykernel_15244\2246380887.py:7: DeprecationWarning: 'asyncio.WindowsProactorEventLoopPolicy' is deprecated and slated for removal in Python 3.16
  if not isinstance(asyncio.get_event_loop_policy(), asyncio.WindowsProactorEventLoopPolicy):
C:\Users\Najeeb\AppData\Local\Temp\ipykernel_15244\2246380887.py:8: DeprecationWarning: 'asyncio.WindowsProactorEventLoopPolicy' is deprecated and slated for removal in Python 3.16
  asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
C:\Users\Najeeb\AppData\Local\Temp\ipykernel_15244\2246380887.py:8: DeprecationWarning: 'asyncio.set_event_loop_policy' is deprecated and slated for removal in Python 3.16
  asyncio.set_event_loop_policy(

# Local MCP Server

In [2]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "local_server": {
            "transport": "stdio",
            "command": "python",
            "args": ["resources/2.1_mcp_server.py"]
        }
    }
)

In [9]:
tools = await client.get_tools()

resource = await client.get_resources("local_server")

prompt = await client.get_prompt("local_server", "prompt")

from pprint import pprint

p_content = prompt[0].content

In [10]:
from langchain_ollama.chat_models import ChatOllama

model = ChatOllama(model="qwen3.5")

from langchain.agents import create_agent

agent = create_agent(model=model, tools=tools, system_prompt=p_content)

In [11]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = await agent.ainvoke(
    {"messages": [HumanMessage(content="Tell me about the langchain-mcp-adapters library")]},
    config=config
)
print(response)

{'messages': [HumanMessage(content='Tell me about the langchain-mcp-adapters library', additional_kwargs={}, response_metadata={}, id='c345ac68-07d8-4537-b21c-2f827d3f7b09'), AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3.5', 'created_at': '2026-04-03T06:08:58.1303359Z', 'done': True, 'done_reason': 'stop', 'total_duration': 5252220900, 'load_duration': 4310460700, 'prompt_eval_count': 417, 'prompt_eval_duration': 158257000, 'eval_count': 71, 'eval_duration': 679578200, 'logprobs': None, 'model_name': 'qwen3.5', 'model_provider': 'ollama'}, id='lc_run--019d51f5-708b-7433-9fde-8e8a359c45bf-0', tool_calls=[{'name': 'search_web', 'args': {'query': 'langchain-mcp-adapters library'}, 'id': '90ae8c37-99ba-4999-860c-8bcf5344c1e4', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 417, 'output_tokens': 71, 'total_tokens': 488}), ToolMessage(content=[{'type': 'text', 'text': "LangChainMCPAdaptersThislibraryprovides a lightweight wrappe

In [12]:
pprint(response)

{'messages': [HumanMessage(content='Tell me about the langchain-mcp-adapters library', additional_kwargs={}, response_metadata={}, id='c345ac68-07d8-4537-b21c-2f827d3f7b09'),
              AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3.5', 'created_at': '2026-04-03T06:08:58.1303359Z', 'done': True, 'done_reason': 'stop', 'total_duration': 5252220900, 'load_duration': 4310460700, 'prompt_eval_count': 417, 'prompt_eval_duration': 158257000, 'eval_count': 71, 'eval_duration': 679578200, 'logprobs': None, 'model_name': 'qwen3.5', 'model_provider': 'ollama'}, id='lc_run--019d51f5-708b-7433-9fde-8e8a359c45bf-0', tool_calls=[{'name': 'search_web', 'args': {'query': 'langchain-mcp-adapters library'}, 'id': '90ae8c37-99ba-4999-860c-8bcf5344c1e4', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 417, 'output_tokens': 71, 'total_tokens': 488}),
              ToolMessage(content=[{'type': 'text', 'text': "LangChainMCPAdaptersThislibraryp

In [13]:
client = MultiServerMCPClient(
    {
        "time": {
            "transport": "stdio",
            "command": "uvx",
            "args": [
                "mcp-server-time",
                "--local-timezone=America/New_York"
            ]
        }
    }
)

tools = await client.get_tools()

In [14]:
agent = create_agent(
    model=model,
    tools=tools
)

In [15]:
question = HumanMessage(content="What time is it?")

response = await agent.ainvoke(
    {"messages": [question]}
)

pprint(response)

{'messages': [HumanMessage(content='What time is it?', additional_kwargs={}, response_metadata={}, id='cca0e85b-5ef5-4d3a-8ac7-e2abacc1fbc2'),
              AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3.5', 'created_at': '2026-04-03T06:13:03.0295628Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1512860800, 'load_duration': 120736000, 'prompt_eval_count': 511, 'prompt_eval_duration': 318948400, 'eval_count': 107, 'eval_duration': 1019392900, 'logprobs': None, 'model_name': 'qwen3.5', 'model_provider': 'ollama'}, id='lc_run--019d51f9-3bcc-70a2-8a8f-1505750ff2b1-0', tool_calls=[{'name': 'get_current_time', 'args': {'timezone': 'America/New_York'}, 'id': '8fd25f38-4a82-4068-b4bc-da695b0c4be6', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 511, 'output_tokens': 107, 'total_tokens': 618}),
              ToolMessage(content=[{'type': 'text', 'text': '{\n  "timezone": "America/New_York",\n  "datetime": "2026-04-03T02: